# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract answers 1-3 (plain words):**

1. **What one row means.** One content item (a `content_hash_id` inside a `client_hash_id`), with its search numbers added up over a half-month window. The source table is daily (one row per date x client x content), so I collapse it to one row per item.
2. **Which table.** `fact_content_daily_performance`, one month partition only (`month=2026-03`). Every number comes from it. `dim_clients` (per-client history start) is for context in a later week; I do not query it this week.
3. **Which time window.** A mid-panel month, March 2026. The decision moment is the end of 15 March. Features use 1-15 March. The label proxy uses 16-31 March. June 2026 (the final month) is sealed and not touched.

The code cell below is setup only: it connects to the release and reads the table's columns, so the checks in section 3 use the real column names. The token comes from a Colab Secret named `HF_TOKEN` and is never printed or typed into a cell (this repo is public).

In [ ]:
%pip -q install duckdb

import os, getpass
import duckdb
import numpy as np
import pandas as pd

# Token order: env var -> Colab Secret named HF_TOKEN -> prompt (last resort).
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"          # mid-panel development month; the final month (2026-06) stays sealed
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Setup (not one of the three checks): read the column list, find the GSC availability flag.
schema = con.sql(f"DESCRIBE SELECT * FROM {FACT}").df()[["column_name", "column_type"]]
print(schema.to_string(index=False))
cols = list(schema["column_name"])

need = ["report_date", "client_hash_id", "content_hash_id",
        "gsc_impressions", "gsc_clicks", "gsc_avg_position"]
missing = [c for c in need if c not in cols]
assert not missing, f"Columns not found: {missing} - read the schema above and fix the names"

GSC_AVAIL = next((c for c in cols if "gsc" in c and "avail" in c), None)
assert GSC_AVAIL, "No GSC availability column found - read the schema above and set GSC_AVAIL by hand"
print("\nAvailability flag I will filter on:", GSC_AVAIL)

Paste your Hugging Face READ token (hf_...): ··········
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_g

## 2. Fields: feature / label / context / excluded

**Contract answers 4-5 (plain words):**

4. **What I rank, and the proxy behind it.** I want a ranked list of pages that are likely to keep under-capturing clicks for their position. The label I can build is a **proxy**: `low_later` = 1 when a page's CTR in 16-31 March is under half of what pages in the same position tier got in that same window. The clicks and impressions are observed; the "half of tier typical" cut-off and the tiers are my own definitions. There is no observed "page was fixed and clicks recovered" outcome in this data.
5. **One thing I deliberately exclude.** Anything from 16-31 March as a feature, and anything computed from it (including that window's CTR). That window is the label window; using it would hand the model the answer.

| Bucket | Fields | Why |
|---|---|---|
| **Feature** (built in section 3) | `imp_h1`, `ctr_h1`, `pos_h1`, `active_days_h1`, `gap_h1` | All computed from 1-15 March only |
| **Label / proxy** | `low_later` (from `clk_h2`, `imp_h2`, `ctr_h2`) | Computed from 16-31 March; never a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Grouping, joining and splitting only; IDs are pseudonyms |
| **Excluded** | Every 16-31 March column used as a feature | Label window |
| **Excluded** | GA4 columns | Not needed for search CTR; early rows are zero-filled, not "no engagement" |
| **Excluded** | `fact_content_query_90d` | Its 90-day window is the most recent months of the release, which is after March, so it is future information for a 15 March decision |

In [ ]:
CONTRACT = {
    "row": "one content item (content_hash_id within client_hash_id), half-month aggregates",
    "table": "fact_content_daily_performance, partition month=2026-03",
    "feature_window": "2026-03-01 .. 2026-03-15",
    "label_window": "2026-03-16 .. 2026-03-31",
    "label_proxy": "low_later = CTR in label window < 0.5 x typical CTR of the page's position tier",
    "excluded": "anything from the label window as a feature",
}
for k, v in CONTRACT.items():
    print(f"{k:15} {v}")

row             one content item (content_hash_id within client_hash_id), half-month aggregates
table           fact_content_daily_performance, partition month=2026-03
feature_window  2026-03-01 .. 2026-03-15
label_window    2026-03-16 .. 2026-03-31
label_proxy     low_later = CTR in label window < 0.5 x typical CTR of the page's position tier
excluded        anything from the label window as a feature


## 3. Verify it with queries (grain, counts, missing values, windows)

Three checks, each on the March 2026 partition. Each one has a one-line "what I expect" so the output can be compared against the claim above. Expect the first one to take a few minutes in Colab (it reads a month of daily rows over the network).

**Check 1 - grain.** My claim: one row in the source is one (date, client, content). A group with more than one row would break that. Zero rows back means the grain holds.

In [ ]:
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate (date, client, content) groups found:", len(grain))
grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) groups found: 0


,report_date,client_hash_id,content_hash_id,c


**Check 2 - slice size and date span.** My claim: the partition covers 1-31 March 2026. I want the row count, how many clients and items, and the first and last date.

In [ ]:
span = con.sql(f"""
    SELECT COUNT(*)                       AS rows_in_slice,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date)                AS first_date,
           MAX(report_date)                AS last_date
    FROM {FACT}
""").df()
span

,rows_in_slice,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


**Check 3 - availability.** The release flags are three-valued (TRUE, FALSE, NULL), so `= FALSE` or `NOT ...` would silently mishandle the NULLs. I filter with `IS TRUE` and show how many rows survive.

In [ ]:
avail = con.sql(f"""
    SELECT COUNT(*)                                                    AS total_rows,
           SUM(CASE WHEN {GSC_AVAIL} IS TRUE     THEN 1 ELSE 0 END)    AS is_true,
           SUM(CASE WHEN {GSC_AVAIL} IS NOT TRUE THEN 1 ELSE 0 END)    AS is_not_true,
           SUM(CASE WHEN {GSC_AVAIL} IS NULL     THEN 1 ELSE 0 END)    AS is_null
    FROM {FACT}
""").df()
avail["share_surviving_IS_TRUE"] = (avail["is_true"] / avail["total_rows"]).round(3)
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,is_true,is_not_true,is_null,share_surviving_IS_TRUE
0,9841378,3611061.0,6230317.0,0.0,0.367


### Five features

I build one frame, one row per content item, from the same month. Rows must pass `IS TRUE` on the availability flag and have a real position (`gsc_avg_position > 0`; zero means "no data"). I keep items with at least 100 impressions in **both** halves so CTR is not decided by one click. The 16-31 March columns (`imp_h2`, `clk_h2`) are kept only to build the label; they are not features.

In [ ]:
D1 = "DATE '2026-03-15'"   # decision moment: end of 15 March
D2 = "DATE '2026-03-16'"   # label window starts

frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= {D1} THEN gsc_impressions END) AS imp_h1,
        SUM(CASE WHEN report_date <= {D1} THEN gsc_clicks END)      AS clk_h1,
        SUM(CASE WHEN report_date <= {D1} THEN gsc_avg_position * gsc_impressions END)
            / NULLIF(SUM(CASE WHEN report_date <= {D1} THEN gsc_impressions END), 0) AS pos_h1,
        COUNT(DISTINCT CASE WHEN report_date <= {D1} AND gsc_impressions > 0 THEN report_date END) AS active_days_h1,
        SUM(CASE WHEN report_date >= {D2} THEN gsc_impressions END) AS imp_h2,
        SUM(CASE WHEN report_date >= {D2} THEN gsc_clicks END)      AS clk_h2
    FROM {FACT}
    WHERE {GSC_AVAIL} IS TRUE AND gsc_avg_position > 0
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_h1 >= 100 AND imp_h2 >= 100
""").df()

# Feature-window features (all knowable on 15 March)
frame["ctr_h1"] = 100 * frame["clk_h1"] / frame["imp_h1"]          # percent units
frame["tier"] = pd.cut(frame["pos_h1"], bins=[0, 3, 10, 20, 50, np.inf],
                       labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
t1 = frame.groupby("tier", observed=True)[["clk_h1", "imp_h1"]].sum()
bench_h1 = 100 * t1["clk_h1"] / t1["imp_h1"]                      # typical CTR per tier, 1-15 March only
frame["gap_h1"] = frame["ctr_h1"] - frame["tier"].map(bench_h1).astype(float)

# Label proxy (16-31 March) - kept out of FEATURES
frame["ctr_h2"] = 100 * frame["clk_h2"] / frame["imp_h2"]
t2 = frame.groupby("tier", observed=True)[["clk_h2", "imp_h2"]].sum()
bench_h2 = 100 * t2["clk_h2"] / t2["imp_h2"]
frame["low_later"] = (frame["ctr_h2"] < 0.5 * frame["tier"].map(bench_h2).astype(float)).astype(int)

FEATURES = ["imp_h1", "ctr_h1", "pos_h1", "active_days_h1", "gap_h1"]
assert len(FEATURES) == 5
assert not any(c.endswith("_h2") or c == "low_later" for c in FEATURES), "a label-window column slipped into FEATURES"

print("Rows in frame (one row = one content item):", len(frame))
print("Duplicate (client, content) rows:", frame.duplicated(["client_hash_id", "content_hash_id"]).sum())
print(f"Label base rate (low_later = 1): {frame['low_later'].mean():.1%}")
frame[["client_hash_id", "content_hash_id"] + FEATURES + ["low_later"]].head(8).round(2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in frame (one row = one content item): 71628
Duplicate (client, content) rows: 0
Label base rate (low_later = 1): 52.1%


,client_hash_id,content_hash_id,imp_h1,ctr_h1,pos_h1,active_days_h1,gap_h1,low_later
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,215.0,0.47,4.44,14,0.13,1
1,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,0.24,4.66,15,-0.09,1
2,client_62f4a7e64f5e0096,content_225dc9235023be5f,279.0,0.36,10.85,15,0.02,1
3,client_62f4a7e64f5e0096,content_26f5092ee7f70d45,2553.0,0.00,7.51,15,-0.33,1
4,client_62f4a7e64f5e0096,content_cfad137c1b04251b,183.0,0.00,6.85,15,-0.33,1
5,client_62f4a7e64f5e0096,content_9e7c70abfbae371e,1676.0,0.00,5.66,15,-0.33,1
6,client_62f4a7e64f5e0096,content_c0fc3b40ce00a5d7,157.0,0.64,1.57,13,0.20,1
7,client_62f4a7e64f5e0096,content_d79c0df3e1b437fb,142.0,0.00,1.82,13,-0.44,1


**Each feature, and when it is knowable (decision moment = end of 15 March):**

- `imp_h1` - total impressions 1-15 March. Knowable because it only adds up days that are already over.
- `ctr_h1` - clicks divided by impressions over 1-15 March, in percent. Knowable for the same reason; both inputs end on 15 March.
- `pos_h1` - impression-weighted average position over 1-15 March. Knowable because it uses only first-half rows (the weighting stops one big day from setting it).
- `active_days_h1` - number of days in 1-15 March with any impressions. Knowable because it counts days that already happened; it tells a steady page from a spiky one.
- `gap_h1` - `ctr_h1` minus the typical CTR of the page's position tier in 1-15 March. Knowable because the benchmark is built from first-half data only, never from the label window.

### The trap (leakage on purpose)

I score a quick model on a split that holds out **whole clients** (rows from one client never sit on both sides). Then I add one label-derived column, `ctr_h2` (the CTR in 16-31 March, the very number the label is built from), and watch the score jump. Then I delete it and keep the honest number.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

work = frame.sample(min(len(frame), 100_000), random_state=0).reset_index(drop=True)

def quick_score(cols):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr, te = next(gss.split(work, work["low_later"], groups=work["client_hash_id"]))
    model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(work.iloc[tr][cols], work.iloc[tr]["low_later"])
    p = model.predict_proba(work.iloc[te][cols])[:, 1]
    return roc_auc_score(work.iloc[te]["low_later"], p)

# 1) the deliberate leak: add one label-derived column
work["ctr_h2_LEAK"] = work["ctr_h2"]
leaky = quick_score(FEATURES + ["ctr_h2_LEAK"])

# 2) delete it, keep the honest number
work = work.drop(columns=["ctr_h2_LEAK"])
assert "ctr_h2_LEAK" not in work.columns
honest = quick_score(FEATURES)

# one simple reference: the single column gap_h1 alone (lower gap = more likely low later)
split = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
             .split(work, work["low_later"], groups=work["client_hash_id"]))[1]
gap_only = roc_auc_score(work.iloc[split]["low_later"], -work.iloc[split]["gap_h1"])

print(f"AUC with the leaked column:       {leaky:.3f}   <- too good: it contains the answer")
print(f"AUC after deleting it (honest):   {honest:.3f}")
print(f"AUC of gap_h1 alone (reference):  {gap_only:.3f}")

AUC with the leaked column:       1.000   <- too good: it contains the answer
AUC after deleting it (honest):   0.795
AUC of gap_h1 alone (reference):  0.731


**How to read the three numbers:** the leaked score should sit close to 1.0 because `ctr_h2` is what the label is made from; that is the lesson, not a result. The honest number is the one I keep. If it is only a little above the `gap_h1`-alone line, then the five-feature model has not yet earned its place over the simple position-adjusted gap, and I say that plainly in later weeks. AUC here is a quick check, not my final metric (precision@K comes later).

## 4. Data limits

**One limitation I am naming: the panel is unbalanced and client-concentrated.** Clients do not have the same amount of history or the same number of pages, so a few clients can hold most of the rows in this slice. A random split would let a model memorise one client's habits and look better than it is, which is why my quick score holds out whole clients. The cell below prints the real concentration for this slice.

Other limits I already know about, in short: this is one month, so it says nothing about seasonality; each window is only 15 days, so CTR on low-volume pages is noisy (hence the 100-impression floor); the label is a proxy I defined, not an observed outcome; and the final month (June 2026) is sealed, so none of this has been tested on it. What this data can never tell me is *why* a page's CTR is low or whether a rewrite would fix it.

In [ ]:
per_client = frame.groupby("client_hash_id").size().sort_values(ascending=False)
print(f"Clients in the frame:        {len(per_client)}")
print(f"Largest client's share:      {per_client.iloc[0] / len(frame):.0%} of rows")
print(f"Top 3 clients' share:        {per_client.head(3).sum() / len(frame):.0%} of rows")
print(f"Median client holds:         {int(per_client.median())} rows")

Clients in the frame:        36
Largest client's share:      23% of rows
Top 3 clients' share:        55% of rows
Median client holds:         417 rows
